In [2]:
import pandas as pd
import os

In [8]:
df = pd.read_csv('../../data/raw_data/cardekho_listings_delhi.csv')

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7536 entries, 0 to 7535
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Title        7536 non-null   str  
 1   Link         7536 non-null   str  
 2   Price        7530 non-null   str  
 3   Information  7536 non-null   str  
 4   Image        7535 non-null   str  
 5   Year         7536 non-null   int64
 6   Brand        7536 non-null   str  
 7   Location     7536 non-null   str  
dtypes: int64(1), str(7)
memory usage: 471.1 KB


In [19]:
df.columns

Index(['Title', 'Link', 'Price', 'Information', 'Image', 'Year', 'Brand',
       'Location'],
      dtype='str')

In [14]:
df.head()

,Title,Link,Price,Information,Image,Year,Brand,Location
0,2014 Maruti Suzuki Celerio VXI AT,https://www.cardekho.com/used-car-details/used...,"₹ 231,000","60,000 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/49941...,2014,Maruti,"Badshahpur, Gurgaon"
1,2019 Maruti Suzuki Ciaz Zeta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","58,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/51632...,2019,Maruti,"Lajpat Nagar, New Delhi"
2,2018 Maruti Suzuki Ciaz Delta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","41,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/50705...,2018,Maruti,"Lajpat Nagar, New Delhi"
3,2022 Maruti Suzuki XL6 Alpha AT BSVI,https://www.cardekho.com/buy-used-car-details/...,"₹ 1,092,000","53,366 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/51622...,2022,Maruti,"Indirapuram, Ghaziabad"
4,2019 Maruti Suzuki Ertiga 1.5 VDI,https://www.cardekho.com/used-car-details/used...,"₹ 830,000","1,46,000 kms • Diesel • Manual",https://images10.gaadi.com/usedcar_image/50806...,2019,Maruti,Greater Noida


In [37]:
units = set()

def handle_information(data):
    data = data.replace(' • ', '')
    parts = data.split()
    
    distance = parts[0].replace(',', '')
    unit = parts[1]
    units.add(unit)

    if len(parts) == 2:
        fuel_type,vehicle_type= None,None
    elif len(parts) == 3:
        fuel_type, vehicle_type = parts[2], None
    elif len(parts) == 4:
        fuel_type, vehicle_type = parts[2], parts[3]
    else:
        raise ValueError(f"Unexpected format: {parts}")
    
    return (int(distance), fuel_type, vehicle_type)

df[['Kms Covered', 'Fuel Type', 'Type']] = df['Information'].apply(handle_information).apply(pd.Series)

In [40]:
df.head()

,Title,Link,Price,Information,Image,Year,Brand,Location,Kms Covered,Fuel Type,Type
0,2014 Maruti Suzuki Celerio VXI AT,https://www.cardekho.com/used-car-details/used...,"₹ 231,000","60,000 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/49941...,2014,Maruti,"Badshahpur, Gurgaon",60000.0,Petrol,Automatic
1,2019 Maruti Suzuki Ciaz Zeta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","58,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/51632...,2019,Maruti,"Lajpat Nagar, New Delhi",58000.0,Petrol,Manual
2,2018 Maruti Suzuki Ciaz Delta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","41,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/50705...,2018,Maruti,"Lajpat Nagar, New Delhi",41000.0,Petrol,Manual
3,2022 Maruti Suzuki XL6 Alpha AT BSVI,https://www.cardekho.com/buy-used-car-details/...,"₹ 1,092,000","53,366 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/51622...,2022,Maruti,"Indirapuram, Ghaziabad",53366.0,Petrol,Automatic
4,2019 Maruti Suzuki Ertiga 1.5 VDI,https://www.cardekho.com/used-car-details/used...,"₹ 830,000","1,46,000 kms • Diesel • Manual",https://images10.gaadi.com/usedcar_image/50806...,2019,Maruti,Greater Noida,146000.0,Diesel,Manual


In [41]:
units

{'kms'}

In [45]:
df['Fuel Type'].unique()

<StringArray>
['Petrol', 'Diesel', 'CNG', nan, 'Electric']
Length: 5, dtype: str

In [46]:
df['Type'].unique()

<StringArray>
['Automatic', 'Manual', nan]
Length: 3, dtype: str

In [54]:
df['Price'].isna().sum()

np.int64(6)

In [57]:
df =df.dropna(subset=['Price']).reset_index()

In [59]:
df.drop('index',axis = 1,inplace = True)

In [61]:
def refine_price(data):
    if type(data) == str:
        data = data.replace('₹','').replace(',','')
        return float(data)
    return data

df['Price'] = df['Price'].apply(refine_price)

In [63]:
df.drop('Information',inplace=True,axis = 1)

In [64]:
df.head()

,Title,Link,Price,Image,Year,Brand,Location,Kms Covered,Fuel Type,Type
0,2014 Maruti Suzuki Celerio VXI AT,https://www.cardekho.com/used-car-details/used...,231000.0,https://images10.gaadi.com/usedcar_image/49941...,2014,Maruti,"Badshahpur, Gurgaon",60000.0,Petrol,Automatic
1,2019 Maruti Suzuki Ciaz Zeta BSIV,https://www.cardekho.com/used-car-details/used...,575000.0,https://images10.gaadi.com/usedcar_image/51632...,2019,Maruti,"Lajpat Nagar, New Delhi",58000.0,Petrol,Manual
2,2018 Maruti Suzuki Ciaz Delta BSIV,https://www.cardekho.com/used-car-details/used...,575000.0,https://images10.gaadi.com/usedcar_image/50705...,2018,Maruti,"Lajpat Nagar, New Delhi",41000.0,Petrol,Manual
3,2022 Maruti Suzuki XL6 Alpha AT BSVI,https://www.cardekho.com/buy-used-car-details/...,1092000.0,https://images10.gaadi.com/usedcar_image/51622...,2022,Maruti,"Indirapuram, Ghaziabad",53366.0,Petrol,Automatic
4,2019 Maruti Suzuki Ertiga 1.5 VDI,https://www.cardekho.com/used-car-details/used...,830000.0,https://images10.gaadi.com/usedcar_image/50806...,2019,Maruti,Greater Noida,146000.0,Diesel,Manual


In [65]:
df.to_csv('../../data/cleaned_data/cardekho_cleaned_data.csv')

## Cleaning Cars24 Data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data/raw_data/cars24_listings_delhi.csv')

In [4]:
df.head()

,Title,Link,Price,Del Price,Information,Image,Year,Brand,Location
0,NaN,https://www.cars24.com/buy-used-maruti-new-wag...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN
1,NaN,https://www.cars24.com/buy-used-maruti-wagon-r...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN
2,NaN,https://www.cars24.com/buy-used-maruti-new-wag...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN
3,NaN,https://www.cars24.com/buy-used-maruti-new-wag...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN
4,NaN,https://www.cars24.com/buy-used-maruti-new-wag...,NaN,NaN,NaN,https://media.cars24.com/hello-ar/dev/transfor...,NaN,NaN,NaN


In [8]:
df = df.dropna(subset=['Title']).reset_index()

In [10]:
df.drop(['index'],axis = 1,inplace = True)

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        613 non-null    str    
 1   Link         613 non-null    str    
 2   Price        613 non-null    str    
 3   Del Price    277 non-null    str    
 4   Information  613 non-null    str    
 5   Image        613 non-null    str    
 6   Year         613 non-null    float64
 7   Brand        613 non-null    str    
 8   Location     613 non-null    str    
dtypes: float64(1), str(8)
memory usage: 43.2 KB


In [13]:
df.head()

,Title,Link,Price,Del Price,Information,Image,Year,Brand,Location
0,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 7.72 lakh,"₹ 899,000","45,498 km | Petrol | Manual | HR-98",https://media.cars24.com/hello-ar/dev/transfor...,2022.0,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"
1,2019 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 5.90 lakh,"₹ 591,000","93,355 km | Petrol | Manual | DL-6C",https://media.cars24.com/hello-ar/dev/transfor...,2019.0,Maruti,Rajouri Garden
2,2024 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 8.99 lakh,"₹ 917,000","18,586 km | CNG | Manual | HR-14",https://media.cars24.com/hello-ar/dev/transfor...,2024.0,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"
3,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 8.15 lakh,"₹ 931,000","33,086 km | Petrol | Auto | DL-4C",https://media.cars24.com/hello-ar/dev/transfor...,2022.0,Maruti,"Sector-18, Noida"
4,2016 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,₹ 4.50 lakh,"₹ 487,000","96,696 km | Petrol | Manual | HR-26",https://media.cars24.com/hello-ar/dev/transfor...,2016.0,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"


In [ ]:
def refine_prices(data):
    data = data.replace('₹','')
    if 'lakh' in data:
        return float(data.replace('lakh',''))*100000

df['Price'] = df['Price'].apply(refine_prices)


In [17]:
df['Year'] = df['Year'].astype(int)

In [18]:
df.head()

,Title,Link,Price,Del Price,Information,Image,Year,Brand,Location
0,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,772000.0,"₹ 899,000","45,498 km | Petrol | Manual | HR-98",https://media.cars24.com/hello-ar/dev/transfor...,2022,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"
1,2019 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,590000.0,"₹ 591,000","93,355 km | Petrol | Manual | DL-6C",https://media.cars24.com/hello-ar/dev/transfor...,2019,Maruti,Rajouri Garden
2,2024 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,899000.0,"₹ 917,000","18,586 km | CNG | Manual | HR-14",https://media.cars24.com/hello-ar/dev/transfor...,2024,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"
3,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,815000.0,"₹ 931,000","33,086 km | Petrol | Auto | DL-4C",https://media.cars24.com/hello-ar/dev/transfor...,2022,Maruti,"Sector-18, Noida"
4,2016 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,450000.0,"₹ 487,000","96,696 km | Petrol | Manual | HR-26",https://media.cars24.com/hello-ar/dev/transfor...,2016,Maruti,"M3M Urbana, Golf Course Ext., Gurugram"


In [28]:
l = set()
def extract_information(rec):
    distance,fuel_type,type,region_info = rec.split('|')
    distance = int(distance.replace(',','').replace('km',''))
    if type == ' Auto ':
        type = 'Automatic'
    return distance,fuel_type.strip(),type.strip(),region_info.strip()  

df[['Kms Covered','Fuel Type','Type','Region Info']] = df['Information'].apply(extract_information).apply(pd.Series)

In [29]:
df['Fuel Type'].unique()

<StringArray>
['Petrol', 'CNG', 'Diesel', 'Hybrid']
Length: 4, dtype: str

In [30]:
df['Type'].unique()

<StringArray>
['Manual', 'Automatic']
Length: 2, dtype: str

In [31]:
df['Region Info'].unique()

<StringArray>
['HR-98', 'DL-6C', 'HR-14', 'DL-4C', 'HR-26', 'DL-1C', 'DL-2C', 'UP-16',
 'DL-8C', 'DL-3C', 'DL-9C', 'HR-79', 'UP-80', 'HR-05', 'JH-09', 'HR-76',
 'HR-51', 'DL-12', 'HR-21', 'HR-02', 'UP-61', 'MP-07', 'HR-87', 'HR-10',
 'UP-14', 'DL-11', 'HR-19', 'DL-5C', 'HR-36', 'HR-27', 'DL-10', 'UP-15',
 'DL-7C', 'HR-06', 'HR-01', 'HR-30', 'UP-32', 'HR-50', 'BR-01', 'HR-29',
 'UK-07', 'UP-13', 'HR-33', 'HR-20', 'JK-02', 'UP-37', 'HR-89', 'UP-33',
 'DL-14', 'UP-85', 'HR-44', 'TN-10', 'RJ-32', 'JH-05', 'BR-19', '24-BH',
 'UP-63', 'CH-01', 'GJ-01', 'HR-12', 'HR-49', 'HP-68', 'WB-02', 'HR-70',
 'MH-47', 'GJ-18', 'HR-16', 'HR-85', 'DL-13', 'RJ-13', 'UP-78', 'TS-02',
 'UP-27', 'UP-21']
Length: 74, dtype: str

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        613 non-null    str    
 1   Link         613 non-null    str    
 2   Price        613 non-null    float64
 3   Del Price    277 non-null    str    
 4   Information  613 non-null    str    
 5   Image        613 non-null    str    
 6   Year         613 non-null    int64  
 7   Brand        613 non-null    str    
 8   Location     613 non-null    str    
 9   Kms Covered  613 non-null    int64  
 10  Fuel Type    613 non-null    str    
 11  Type         613 non-null    str    
 12  Region Info  613 non-null    str    
dtypes: float64(1), int64(2), str(10)
memory usage: 62.4 KB


In [33]:
df.drop(['Del Price','Information'],axis = 1,inplace = True)

In [34]:
df.head()

,Title,Link,Price,Image,Year,Brand,Location,Kms Covered,Fuel Type,Type,Region Info
0,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,772000.0,https://media.cars24.com/hello-ar/dev/transfor...,2022,Maruti,"M3M Urbana, Golf Course Ext., Gurugram",45498,Petrol,Manual,HR-98
1,2019 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,590000.0,https://media.cars24.com/hello-ar/dev/transfor...,2019,Maruti,Rajouri Garden,93355,Petrol,Manual,DL-6C
2,2024 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,899000.0,https://media.cars24.com/hello-ar/dev/transfor...,2024,Maruti,"M3M Urbana, Golf Course Ext., Gurugram",18586,CNG,Manual,HR-14
3,2022 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,815000.0,https://media.cars24.com/hello-ar/dev/transfor...,2022,Maruti,"Sector-18, Noida",33086,Petrol,Automatic,DL-4C
4,2016 Maruti Ertiga,https://www.cars24.com/buy-used-maruti-ertiga-...,450000.0,https://media.cars24.com/hello-ar/dev/transfor...,2016,Maruti,"M3M Urbana, Golf Course Ext., Gurugram",96696,Petrol,Manual,HR-26


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        613 non-null    str    
 1   Link         613 non-null    str    
 2   Price        613 non-null    float64
 3   Image        613 non-null    str    
 4   Year         613 non-null    int64  
 5   Brand        613 non-null    str    
 6   Location     613 non-null    str    
 7   Kms Covered  613 non-null    int64  
 8   Fuel Type    613 non-null    str    
 9   Type         613 non-null    str    
 10  Region Info  613 non-null    str    
dtypes: float64(1), int64(2), str(8)
memory usage: 52.8 KB


In [37]:
df.to_csv('../../data/cleaned_data/cars24_cleaned_data.csv')

## Olx Data Cleaning

In [39]:
df = pd.read_csv('../../data/raw_data/olx_car_listings_delhi.csv')

In [40]:
df.head()

,Title,Link,Location,Price,Information,Image,Brand
0,Skoda Slavia,https://www.olx.in/item/cars-c84-used-skoda-sl...,Bali Nagar,"₹ 12,50,000","2023 - 45,000 km",https://apollo.olx.in:443/v1/files/4e0x48rz3m5...,maruti suzuki
1,Ford Ecosport,https://www.olx.in/item/cars-c84-used-ford-eco...,Rajouri Garden,"₹ 2,85,000","2014 - 65,000 km",https://apollo.olx.in:443/v1/files/mb5jfzv57gr...,maruti suzuki
2,Ford Figo,https://www.olx.in/item/cars-c84-used-ford-fig...,Rajouri Garden,"₹ 3,75,000","2019 - 42,000 km",https://apollo.olx.in:443/v1/files/2vftmr54s5h...,maruti suzuki
3,Hyundai Creta,https://www.olx.in/item/cars-c84-used-hyundai-...,Subhash Nagar,"₹ 4,99,999","2016 - 60,000 km",https://apollo.olx.in:443/v1/files/6c1jfi8ai1b...,maruti suzuki
4,Mahindra Thar,https://www.olx.in/item/cars-c84-used-mahindra...,Rajouri Garden,"₹ 12,25,000","2024 - 30,000 km",https://apollo.olx.in:443/v1/files/9nw3e2jxq89...,maruti suzuki


In [ ]:
def filter_price(data):
        data.replace(',','').replace(',','')
